In [ ]:
import os
import pandas as pd
import numpy as np
import pywt
#This script denoises stock log returns using wavelet transforms and saves the cleaned data.
def denoise_with_wavelet(returns, wavelet, level):
    """Denoise returns using SWT + universal thresholding at given wavelet and level."""

    # ensure length is divisible by 2**level
    L = 2 ** level
    n = len(returns)
    if n % L != 0:
        returns = returns[:n - (n % L)]  

    coeffs = pywt.swt(returns, wavelet, level=level)


    sigma = np.median(np.abs(coeffs[0][1])) / 0.6745
    threshold = sigma * np.sqrt(2 * np.log(len(returns)))

    denoised_coeffs = []
    for cA, cD in coeffs:
        cD_thresh = pywt.threshold(cD, threshold, mode='soft')
        denoised_coeffs.append((cA, cD_thresh))

    # Reconstruct
    smoothed = pywt.iswt(denoised_coeffs, wavelet)
    return smoothed

def process_stocks(tickers, input_dir="data", output_dir="clean_data"):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    results = []  # to store summary table

    for ticker in tickers:
        input_path = os.path.join(input_dir, f"{ticker}.csv")
        if not os.path.exists(input_path):
            print(f"Skipping {ticker} — file not found in {input_dir}/")
            continue

        # Load data
        df = pd.read_csv(input_path)
        if "Log_Return" not in df.columns:
            print(f"Skipping {ticker} — Log_Return column missing.")
            continue

        returns = df["Log_Return"].values
        orig_var = np.var(returns)

        best_var = float("inf")
        best_series, best_wavelet, best_level = None, None, None

        for wavelet in ["haar", "coif5", "bior6.8"]:
            for level in [1, 2]:
                try:
                    denoised = denoise_with_wavelet(returns, wavelet, level)
                    denoised_var = np.var(denoised)

                    if 0 < denoised_var < best_var:
                        best_var = denoised_var
                        best_series = denoised
                        best_wavelet = wavelet
                        best_level = level
                except Exception as e:
                    print(f"Error denoising {ticker} with {wavelet} L{level}: {e}")

        if best_series is not None:
            aligned_dates = df["Date"].iloc[:len(best_series)].reset_index(drop=True)
            out_df = pd.DataFrame({"Date": aligned_dates, "Log_Return_Clean": best_series})
            out_path = os.path.join(output_dir, f"{ticker}_clean.csv")
            out_df.to_csv(out_path, index=False)

            results.append({
                "Ticker": ticker,
                "Original_Variance": orig_var,
                "Best_Variance": best_var,
                "Best_Wavelet": best_wavelet,
                "Best_Level": best_level
            })

            print(f"Saved cleaned data for {ticker} → {out_path}")
        else:
            print(f"No valid denoised result for {ticker}")


    summary_df = pd.DataFrame(results)
    summary_path = os.path.join(output_dir, "denoising_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"\nSummary table saved to {summary_path}")
    print(summary_df)


def main():
    tickers = [
        'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'META', 'NVDA', 'PYPL', 'CSCO', 'INTC',
        'AMD', 'NFLX', 'ADBE', 'INTU', 'QCOM', 'TXN', 'AMAT', 'MU', 'KLAC', 'LRCX'
    ]
    process_stocks(tickers)


if __name__ == "__main__":
    main()


Saved cleaned data for AAPL → clean_data/AAPL_clean.csv
Saved cleaned data for MSFT → clean_data/MSFT_clean.csv
Saved cleaned data for GOOGL → clean_data/GOOGL_clean.csv
Saved cleaned data for AMZN → clean_data/AMZN_clean.csv
Saved cleaned data for TSLA → clean_data/TSLA_clean.csv
Saved cleaned data for META → clean_data/META_clean.csv
Saved cleaned data for NVDA → clean_data/NVDA_clean.csv
Skipping PYPL — file not found in data/
Saved cleaned data for CSCO → clean_data/CSCO_clean.csv
Saved cleaned data for INTC → clean_data/INTC_clean.csv
Skipping AMD — file not found in data/
Saved cleaned data for NFLX → clean_data/NFLX_clean.csv
Saved cleaned data for ADBE → clean_data/ADBE_clean.csv
Saved cleaned data for INTU → clean_data/INTU_clean.csv
Saved cleaned data for QCOM → clean_data/QCOM_clean.csv
Saved cleaned data for TXN → clean_data/TXN_clean.csv
Saved cleaned data for AMAT → clean_data/AMAT_clean.csv
Saved cleaned data for MU → clean_data/MU_clean.csv
Saved cleaned data for KLAC →